# Inference Pipeline
## Torsion Flow → NeRF → CoordRefiner → PDB

End-to-end structure generation for a protein sequence:

```
sequence
   ↓  ESM-2 (per-residue embeddings)
   ↓  ConditionedFlow ODE  →  φ/ψ torsion angles
   ↓  NeRF reconstruction  →  backbone coords (L, 4, 3)
   ↓  CoordRefiner         →  refined coords  (L, 4, 3)
   ↓  write_pdb            →  output.pdb
```

Proteins are loaded from the preprocessed HDF5 dataset (`proteins.h5`) produced by `preprocess_dataset.py`.

## 1. Imports & Config

In [ ]:
import os
import random
import pickle as pkl

import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from dataset import ProteinDataset, build_flow_tensors
from models import ConditionedFlow, CoordRefiner
from nerf import build_backbone
from geometry import center_coords, kabsch_align, ca_rmsd
from pipeline.pipeline import generate_and_refine

H5_PATH        = 'proteins.h5'
FLOW_CKPT      = 'flow_model.pt'
REFINER_CKPT   = 'coord_refiner.pt'
DEVICE         = 'cpu'
RANDOM_SEED    = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print('Ready.')

## 2. Load Dataset (HDF5)

Reads from `proteins.h5` produced by `preprocess_dataset.py`. Splits proteins into train / test.

In [ ]:
dataset = ProteinDataset(H5_PATH, min_len=5, max_len=2048)
print(f'Dataset: {len(dataset)} proteins')

# 90 / 10 split
n_test     = max(1, len(dataset) // 10)
all_pids   = dataset.pids
test_pids  = set(random.sample(all_pids, n_test))
train_pids = [p for p in all_pids if p not in test_pids]

print(f'Train: {len(train_pids)}   Test: {len(test_pids)}')

## 3. Flow Model — Train or Load

Trains a `ConditionedFlow` on interior-residue φ/ψ angles conditioned on ESM-2 embeddings.  
Skip training and load from checkpoint if `flow_model.pt` already exists.

In [ ]:
flow = ConditionedFlow(dim=2, embed_dim=320, h=512).to(DEVICE)

if os.path.exists(FLOW_CKPT):
    flow.load_state_dict(torch.load(FLOW_CKPT, map_location=DEVICE))
    flow.eval()
    print(f'Loaded flow model from {FLOW_CKPT}')
else:
    print('Building flow training tensors from HDF5...')
    angles_tensor, esm2_tensor, _ = build_flow_tensors(dataset)
    # restrict to train proteins only
    train_set = set(train_pids)
    train_mask = torch.tensor([
        pid in train_set
        for pid in _   # _ is pdb_ids list from build_flow_tensors
    ])
    angles_train = angles_tensor[train_mask]
    esm2_train   = esm2_tensor[train_mask]
    print(f'Flow training residues: {len(angles_train):,}')

    FLOW_ITERS      = 30_000
    FLOW_BATCH      = 1024
    FLOW_LR         = 1e-3

    optimizer = torch.optim.Adam(flow.parameters(), lr=FLOW_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FLOW_ITERS)
    loss_fn   = nn.MSELoss()
    n         = angles_train.shape[0]
    flow_losses = []

    flow.train()
    for i in range(FLOW_ITERS):
        idx  = torch.randint(0, n, (FLOW_BATCH,))
        x_1  = angles_train[idx]
        emb  = esm2_train[idx]
        x_0  = torch.randn_like(x_1)
        t    = torch.rand(FLOW_BATCH, 1)
        x_t  = (1 - t) * x_0 + t * x_1
        dx_t = x_1 - x_0

        optimizer.zero_grad()
        loss = loss_fn(flow(t, x_t, emb), dx_t)
        loss.backward()
        optimizer.step()
        scheduler.step()
        flow_losses.append(loss.item())

        if (i + 1) % 5000 == 0:
            print(f'  iter {i+1:6d} | loss {loss.item():.4f} | lr {scheduler.get_last_lr()[0]:.2e}')

    flow.eval()
    torch.save(flow.state_dict(), FLOW_CKPT)
    print(f'Flow training complete. Saved → {FLOW_CKPT}')

    plt.figure(figsize=(9, 3))
    w = max(1, FLOW_ITERS // 200)
    plt.plot(np.convolve(flow_losses, np.ones(w)/w, mode='valid'))
    plt.xlabel('Iteration'); plt.ylabel('MSE Loss')
    plt.title('Flow Model Training Loss')
    plt.tight_layout(); plt.show()

## 4. Generate Flow Predictions on All Training Proteins

Run the trained flow model once on every training protein to produce realistic noisy φ/ψ angles.  
Then apply NeRF to reconstruct backbone coordinates and Kabsch-align them to the true structure.

The resulting `(noisy_coords, true_coords, esm2)` triples are stored in memory and used directly
to train the CoordRefiner — this is more faithful than Gaussian noise because the refiner learns
to correct the actual distribution of errors the flow model makes.

In [ ]:
FLOW_STEPS = 20   # ODE steps per protein (20 is fast; use 50 for higher quality)

flow.eval()
training_triples = []   # list of (noisy_coords (L,12), true_coords (L,12), esm2 (L,320))
skipped = 0

with torch.no_grad():
    for pid in tqdm(train_pids, desc='Flow → NeRF'):
        sequence = dataset.get_sequence(pid)
        true_raw = dataset.get_true_coords(pid).astype(np.float64)   # (L, 4, 3)
        esm2_np  = dataset.get_esm2(pid)                              # (L, 320)
        L        = len(sequence)

        if len(true_raw) != L or len(esm2_np) != L:
            skipped += 1
            continue

        # ── Flow ODE → φ/ψ ──────────────────────────────────────────────────
        emb_t = torch.tensor(esm2_np, dtype=torch.float32)
        x     = torch.randn(L, 2)
        ts    = torch.linspace(0., 1., FLOW_STEPS + 1)
        for i in range(FLOW_STEPS):
            t_s = ts[i].view(1, 1).expand(L, 1)
            dt  = (ts[i + 1] - ts[i]).float()
            v_s = flow(t_s, x, emb_t)
            v_m = flow(t_s + dt / 2, x + v_s * dt / 2, emb_t)
            x   = x + dt * v_m
        phi_psi = x.numpy() * 180.0   # (L, 2) degrees

        # ── NeRF → centre → Kabsch-align to true ────────────────────────────
        true_c, _  = center_coords(true_raw)
        noisy_raw  = build_backbone(sequence, phi_psi)
        noisy_c, _ = center_coords(noisy_raw)
        noisy_aln  = kabsch_align(noisy_c, true_c)

        training_triples.append((
            torch.tensor(noisy_aln.reshape(L, 12), dtype=torch.float32),
            torch.tensor(true_c.reshape(L, 12),    dtype=torch.float32),
            emb_t,
        ))

print(f'Generated {len(training_triples)} training triples  ({skipped} skipped)')

# Quick sanity check — distribution of pre-refinement Cα RMSDs
rmsds = [
    ca_rmsd(nc.numpy().reshape(-1, 4, 3), tc.numpy().reshape(-1, 4, 3))
    for nc, tc, _ in training_triples
]
print(f'Pre-refinement Cα RMSD — mean: {np.mean(rmsds):.2f} Å  '
      f'median: {np.median(rmsds):.2f} Å  max: {np.max(rmsds):.2f} Å')

## 5. Train CoordRefiner

The refiner sees the flow-generated NeRF coordinates as input and the true PDB coordinates as target.  
One protein per gradient step; variable-length sequences require no padding.

In [ ]:
REFINER_EPOCHS = 30
REFINER_LR     = 3e-4
PRINT_EVERY    = 5

refiner = CoordRefiner(
    coord_dim=12, esm2_dim=320, d_model=256,
    nhead=8, num_layers=4, ffn_dim=1024, dropout=0.1,
)
n_params = sum(p.numel() for p in refiner.parameters())
print(f'CoordRefiner — {n_params:,} parameters')

optimizer = torch.optim.Adam(refiner.parameters(), lr=REFINER_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=REFINER_EPOCHS * len(training_triples)
)

refiner_losses = []

for epoch in range(REFINER_EPOCHS):
    refiner.train()
    epoch_losses = []
    indices = list(range(len(training_triples)))
    random.shuffle(indices)

    for idx in indices:
        noisy_coords, true_coords, esm2_emb = training_triples[idx]

        optimizer.zero_grad()
        refined = refiner(noisy_coords, esm2_emb)
        loss    = F.mse_loss(refined, true_coords)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(refiner.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_losses.append(loss.item())

    mean_loss = float(np.mean(epoch_losses))
    refiner_losses.extend(epoch_losses)
    if (epoch + 1) % PRINT_EVERY == 0:
        print(f'Epoch {epoch+1:3d}/{REFINER_EPOCHS} | loss {mean_loss:.5f} | '
              f'lr {scheduler.get_last_lr()[0]:.2e}')

torch.save(refiner.state_dict(), REFINER_CKPT)
print(f'\nRefiner training complete. Saved → {REFINER_CKPT}')

plt.figure(figsize=(9, 3))
w = max(1, len(training_triples) // 2)
plt.plot(np.convolve(refiner_losses, np.ones(w)/w, mode='valid'))
plt.xlabel('Protein update step'); plt.ylabel('MSE Loss')
plt.title('CoordRefiner Training Loss (smoothed)')
plt.tight_layout(); plt.show()

## 6. Evaluate on Held-Out Test Proteins

Run the same flow → NeRF → refiner pipeline on each test protein and report Cα RMSD before and after refinement.

In [ ]:
refiner.eval()
flow.eval()

before_rmsds, after_rmsds, lengths = [], [], []

with torch.no_grad():
    for pid in tqdm(sorted(test_pids), desc='Evaluating'):
        sequence = dataset.get_sequence(pid)
        true_raw = dataset.get_true_coords(pid).astype(np.float64)
        esm2_np  = dataset.get_esm2(pid)
        L        = len(sequence)

        if len(true_raw) != L or len(esm2_np) != L:
            continue

        # Flow ODE
        emb_t = torch.tensor(esm2_np, dtype=torch.float32)
        x     = torch.randn(L, 2)
        ts    = torch.linspace(0., 1., FLOW_STEPS + 1)
        for i in range(FLOW_STEPS):
            t_s = ts[i].view(1, 1).expand(L, 1)
            dt  = (ts[i + 1] - ts[i]).float()
            v_s = flow(t_s, x, emb_t)
            v_m = flow(t_s + dt / 2, x + v_s * dt / 2, emb_t)
            x   = x + dt * v_m
        phi_psi = x.numpy() * 180.0

        true_c, _  = center_coords(true_raw)
        noisy_raw  = build_backbone(sequence, phi_psi)
        noisy_c, _ = center_coords(noisy_raw)
        noisy_aln  = kabsch_align(noisy_c, true_c)

        noisy_t  = torch.tensor(noisy_aln.reshape(L, 12), dtype=torch.float32)
        refined_t = refiner(noisy_t, emb_t)

        nc = noisy_aln
        tc = true_c
        rc = refined_t.numpy().reshape(L, 4, 3)

        before_rmsds.append(ca_rmsd(nc, tc))
        after_rmsds.append(ca_rmsd(rc, tc))
        lengths.append(L)

before_arr  = np.array(before_rmsds)
after_arr   = np.array(after_rmsds)
improvement = before_arr - after_arr

print(f'Test proteins: {len(before_arr)}')
print(f'Cα RMSD before refinement : {before_arr.mean():.3f} ± {before_arr.std():.3f} Å')
print(f'Cα RMSD after  refinement : {after_arr.mean():.3f}  ± {after_arr.std():.3f} Å')
print(f'Mean improvement          : {improvement.mean():+.3f} Å  '
      f'({(improvement > 0).mean()*100:.0f}% of proteins improved)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('CoordRefiner Evaluation (flow-generated inputs)', fontsize=13)

axes[0].scatter(before_arr, after_arr, s=30, alpha=0.7, color='steelblue')
lim = max(before_arr.max(), after_arr.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'k--', lw=0.8)
axes[0].set_xlabel('RMSD before (Å)'); axes[0].set_ylabel('RMSD after (Å)')
axes[0].set_title('Before vs After'); axes[0].set_aspect('equal')

axes[1].hist(improvement, bins=20, color='tomato', edgecolor='white')
axes[1].axvline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('RMSD improvement (Å)'); axes[1].set_title('Improvement Distribution')

axes[2].scatter(lengths, before_arr, s=20, alpha=0.6, color='grey',      label='Before')
axes[2].scatter(lengths, after_arr,  s=20, alpha=0.6, color='steelblue', label='After')
axes[2].set_xlabel('Sequence length'); axes[2].set_ylabel('Cα RMSD (Å)')
axes[2].set_title('RMSD vs Length'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

## 7. Full Inference — Generate & Export PDB

Pick any test protein, run the complete pipeline via `generate_and_refine`, and write the refined structure to a PDB file.  
The ESM-2 embeddings are loaded directly from the HDF5 dataset — no ESM-2 model required at inference time.

In [ ]:
example_pid  = sorted(test_pids)[0]
example_seq  = dataset.get_sequence(example_pid)
esm2_cached  = dataset.get_esm2(example_pid)        # (L, 320) — from HDF5
true_raw     = dataset.get_true_coords(example_pid)
true_c, _    = center_coords(true_raw.astype(np.float64))

print(f'Protein : {example_pid}')
print(f'Length  : {len(example_seq)} residues')

out_pdb = f'{example_pid}_refined.pdb'

nerf_coords, refined_coords, phi_psi_deg = generate_and_refine(
    esm2_emb      = esm2_cached,          # precomputed — no ESM-2 model needed
    flow_model    = flow,
    refiner_model = refiner,
    sequence      = example_seq,          # needed for NeRF + PDB atom names
    flow_steps    = FLOW_STEPS,
    output_pdb    = out_pdb,
    device        = DEVICE,
)

rmsd_before = ca_rmsd(nerf_coords,    true_c)
rmsd_after  = ca_rmsd(refined_coords, true_c)

print(f'\nCα RMSD — before refinement : {rmsd_before:.3f} Å')
print(f'Cα RMSD — after  refinement : {rmsd_after:.3f} Å')
print(f'Improvement                 : {rmsd_before - rmsd_after:+.3f} Å')
print(f'\nRefined structure written → {out_pdb}')

# Visualise Cα traces (XY projection)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'{example_pid}  (L={len(example_seq)})', fontsize=13)

def _plot_ca(ax, coords, title):
    ca = coords[:, 1]
    ax.scatter(ca[:, 0], ca[:, 1], c=np.arange(len(ca)), cmap='plasma', s=12, zorder=3)
    ax.plot(ca[:, 0], ca[:, 1], lw=0.5, color='grey', alpha=0.5)
    ax.set_title(title); ax.set_aspect('equal')
    ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)')

_plot_ca(axes[0], true_c,        'True (PDB)')
_plot_ca(axes[1], nerf_coords,   f'NeRF  {rmsd_before:.2f} Å')
_plot_ca(axes[2], refined_coords, f'Refined  {rmsd_after:.2f} Å')
plt.tight_layout(); plt.show()